# Interactive Sharding Exploration — LLaMA-3 8B

This notebook loads a saved `ShardingOptimizer` state and demonstrates
interactive exploration of the sharding strategy space.

**Prerequisites:** Run `examples/example_save_llama3.py` first to generate
the `llama3_8b.ap` file.  No model code, GPU, or process group is needed here.

In [ ]:
from autoparallel.optimize_sharding import ShardingOptimizer

opt = ShardingOptimizer.load("llama3_8b.ap")
print(f"Loaded: {len(opt.nodes)} nodes, mesh {opt.mesh.shape}")

## 1. Visualize the current sharding strategy

In [ ]:
import json

data = opt.get_json()
print(f"Summary: {json.dumps(data['summary'], indent=2)}")
print(f"Mesh: {data['mesh']}")
print(f"Nodes: {len(data['nodes'])}")

In [ ]:
from autoparallel.visualizer.build_display_from_json import generate_visualization_html
from IPython.display import HTML

html = generate_visualization_html(data)
HTML(html)

## 2. Inspect a specific node

In [ ]:
# Find an mm node and inspect its cost breakdown
mm_nodes = [n for n in opt.nodes if 'mm' in n.name or 'einsum' in n.name]
print(f"Found {len(mm_nodes)} matmul/einsum nodes")

if mm_nodes:
    node = mm_nodes[0]
    print(f"\nInspecting: {node.name}")
    opt.print_costs_for_node(node)

## 3. What-if: add a constraint and re-solve

In [ ]:
from torch.distributed.tensor.placement_types import Shard, Replicate

# Save the original solution for comparison
original_solution = opt.get_solution()

# Force a specific node to a different placement
if mm_nodes:
    node = mm_nodes[0]
    print(f"Constraining {node.name} to S(0)R...")
    constraint_names = opt.add_node_constraint(node, (Shard(0), Replicate()))
    print(f"Added constraints: {constraint_names}")

In [ ]:
# Re-solve with the new constraint
new_solution = opt.resolve()
print(f"Re-solved: {len(new_solution)} nodes")

# Compare
opt.diff_solutions(original_solution, new_solution)

In [ ]:
# Revert: remove the constraint and re-solve
opt.remove_constraints(constraint_names)
reverted_solution = opt.resolve()
print("Reverted to unconstrained solution")
opt.diff_solutions(original_solution, reverted_solution)

## 4. Apply prefetch discount and re-solve

In [ ]:
n_modified = opt.apply_prefetch_discount(scale=0.0)
print(f"Discounted {n_modified} decision vars (fully overlapped)")

discounted_solution = opt.resolve()
opt.diff_solutions(original_solution, discounted_solution)

## 5. Export a modified solution for the training script

In [ ]:
# Save placements so they can be loaded in the training script
# via: placements = autop.sharding_optimizer.load_placements("custom_placements.json")
opt.save_placements("custom_placements.json")
print("Saved placements to custom_placements.json")